## 1. Setup

In [ ]:
!pip install ipython==8.12.0

In [ ]:
!pip install datasets
!pip install transformers
!pip install evaluate
!pip install rouge_score
!pip install pycocoevalcap
!pip install sacrebleu


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import TrainingArguments
from transformers import Trainer

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from nltk.translate.nist_score import corpus_nist
import sacrebleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

# from google.colab import drive

torch.manual_seed(110)

In [ ]:
sys.path.append(os.path.abspath('..'))

from e2e_loader import get_e2e_df
from lora_module import inject_lora


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")


In [ ]:
CFG = {
    'model_name': 'gpt2-medium',
    # --- Optimization (as per paper, for E2E) ---
    'max_length': 512,               # --seq_len 512
    'batch_size': 8,                 # --train_batch_size 8
    'grad_accum': 1,                 # --grad_acc 1
    'epochs': 5,                     # --max_epoch 5
    'lr': 0.0002,                    # --lr 0.0002
    'weight_decay': 0.01,            
    'adam_beta2': 0.999,             # --adam_beta2 0.999
    'adam_eps': 1e-8,                
    'warmup_steps': 500,             # --warmup_step 500
    'label_smoothing': 0.1,          
    'grad_clip_norm': 0.0,           # --clip 0.0 (no clipping)
    'use_fp16': True,                # paper trains in fp16
    # --- Inference (as per paper,for E2E) ---
    'beam_size': 10,
    'length_penalty': 0.8,
    'no_repeat_ngram_size': 4,
    'max_new_tokens': 64,
    # --- LoRA (paper) ---
    'lora_rank': 4,                  # --lora_dim 4
    'lora_alpha': 32,                # --lora_alpha 32
    'lora_dropout': 0.1,             
    'lora_targets': ['q', 'v'],      # subset of {'q','k','v','o'}
}

## 2. Prepare Dataset

In [ ]:
train_df = get_e2e_df("trainset.csv")
val_df = get_e2e_df("devset.csv")
testwref_df = get_e2e_df("testset_w_refs.csv")
test_df = get_e2e_df("testset.csv")     # Unused

# print(train_df)

## 3. Tokenizer


In [ ]:
DELIMITER = "<|SEP|>"

def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained('gpt2-medium')

    print("Tokenizer token len - before: ", len(tokenizer))
    tokenizer.add_tokens([DELIMITER])
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print("Tokenizer token len - after: ", len(tokenizer))
    return tokenizer

tokenizer = get_tokenizer()



def tokenize_text(text,padding='max_length'):
    tokens = tokenizer(text, padding=padding, truncation=True, return_tensors="pt", max_length=CFG['max_length'])
    return tokens

In [ ]:
tokenizer.pad_token_id

## 4. Custom Dataset

In [ ]:
class E2EDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        '''
        Returns the length of the dataset.
        '''
        return len(self.data)

    def __getitem__(self, idx):
        '''
        Returns the training/validation/test example as index idx.
        '''

        row = self.data.iloc[idx]
        text = row["text"].strip()
        combined = text
        if len(row) > 1:
            label = row["label"].strip()
            # Append EOS explicitly to supervised target text for cleaner stop behavior.
            combined = text + " " + DELIMITER + " " + label + tokenizer.eos_token

        tokens = tokenize_text(combined)
        input_ids = tokens["input_ids"].squeeze().to(dtype=torch.long)
        attention_mask = tokens["attention_mask"].squeeze().to(dtype=torch.long)
        prompt_text = text + " " + DELIMITER
        prompt_tokens = tokenizer(
            prompt_text,
            truncation=True,
            return_tensors="pt",
            max_length=CFG['max_length'],
            add_special_tokens=False,
        )
        prompt_len = prompt_tokens["input_ids"].shape[-1]

        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        inputs = {
            'input_ids' : input_ids,
            'attention_mask' : attention_mask,
            'labels' : labels,
        }
        
        return inputs


train_dataset = E2EDataset(train_df)
val_dataset = E2EDataset(val_df)
test_dataset = E2EDataset(test_df)

In [ ]:
BATCH_SIZE = CFG["batch_size"]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# batch = next(iter(train_loader))
# print(batch["input_ids"].shape)
# print(batch["labels"].shape)

## 5. LoRA Model

In [ ]:
model_lora = AutoModelForCausalLM.from_pretrained("gpt2-medium").to(device)
# Resize embeddings because delimiter token "||" is newly added to tokenizer vocab.
with torch.no_grad():
    model_lora.resize_token_embeddings(len(tokenizer))
model_lora.config.pad_token_id = tokenizer.pad_token_id

hidden_dim = model_lora.config.hidden_size
print('Hidden size:', hidden_dim, ' Layers:', model_lora.config.n_layer)

In [ ]:
lora_rank = 4
lora_alpha = 32
lora_dropout = 0.1
lora_targets = ['q', 'v']

inject_lora(model_lora, CFG['lora_rank'], CFG['lora_alpha'], CFG['lora_dropout'], hidden_dim, CFG['lora_targets'])

trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_lora.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.4f}%)')

model_lora.to(device);

## 6. Training

In [ ]:
from transformers import TrainerCallback

class GPUMemoryCallback(TrainerCallback):
    def __init__(self):
        self.global_peak = 0.0  # in GB

    def on_epoch_begin(self, args, state, control, **kwargs):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def on_epoch_end(self, args, state, control, **kwargs):
        if torch.cuda.is_available():
            peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
            self.global_peak = max(self.global_peak, peak)
            print(f"Epoch peak: {peak:.2f} GB | Global peak so far: {self.global_peak:.2f} GB")

    def on_train_end(self, args, state, control, **kwargs):
        print(f"Final peak GPU memory across all epochs: {self.global_peak:.2f} GB")

gpu_mem_callback = GPUMemoryCallback()

In [ ]:
OUTPUT_DIR = "/checkpoints/lora/"

# Choose between model_ft or model_lora
model = model_lora

# Trainer by default uses AdamW
training_args = TrainingArguments(
    # output_dir=".",
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=CFG['epochs'],
    learning_rate=CFG['lr'],
    warmup_steps=CFG['warmup_steps'],
    lr_scheduler_type="linear",
    weight_decay=CFG['weight_decay'],
    logging_steps=50,
    save_steps=500,
    label_smoothing_factor=CFG['label_smoothing'],
    adam_beta2=CFG['adam_beta2'],
    max_grad_norm=CFG['grad_clip_norm'],
    gradient_accumulation_steps=CFG['grad_accum'],
    seed=110,
    data_seed=110,
    fp16=CFG['use_fp16'],
    save_strategy = "epoch",
    save_total_limit=1,      
    # save_strategy = "no",
    logging_strategy = "epoch",
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# for tracking peak gpu usage
trainer.add_callback(gpu_mem_callback)

In [ ]:
trainer.train()
# trainer.train(resume_from_checkpoint=OUTPUT_DIR + "checkpoint-10516")
# trainer.train(resume_from_checkpoint=True)

In [ ]:
# # Save just weights
# OUTPUT_DIR = base_dir + "/models/checkpoints-e2e/baseline_3/"

# model = trainer.model
# torch.save(model.state_dict(), OUTPUT_DIR + "model_weights.pt")


In [ ]:
# # Load model from just weights
# model = AutoModelForCausalLM.from_pretrained("gpt2-medium").to(device) # loads config + struct
# tokenizer = get_tokenizer()
# with torch.no_grad():
#     model.resize_token_embeddings(len(tokenizer))
# model.config.pad_token_id = tokenizer.pad_token_id

# model.load_state_dict(torch.load(OUTPUT_DIR + "model_weights.pt"))

In [ ]:
# model_paramcount = count_trainable_params(model)
model_paramcount = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model_paramcount)
print("{} M".format(model_paramcount / 1000000))

## 7. Evaluation

In [ ]:
def data_retrieval(df, unique_mr=False):
        if unique_mr: # for bleu, each unique mr has a list of refs
            mr_ref_mapping = {}
            for idx in range(len(df)):
                row = df.iloc[idx]
                mr = row["text"].strip()
                ref = row["label"].strip()
                
                if mr not in mr_ref_mapping:
                    mr_ref_mapping[mr] = []
                mr_ref_mapping[mr].append(ref)

            mrs = list(mr_ref_mapping.keys())
            refs = list(mr_ref_mapping.values())

            return mrs, refs

        else:   # each mr has a ref, may have duplicate mrs 
            mrs = []
            refs = []
            for idx in range(len(df)):
                row = df.iloc[idx]
                mr = row["text"].strip()
                ref = row["label"].strip()
                mrs.append(mr)
                refs.append(ref)

            return mrs, refs

In [ ]:
class E2EGenerationDataset(Dataset):
    def __init__(self, data: list):
        self.data = data

    def __len__(self):
        '''
        Returns the length of the dataset.
        '''
        return len(self.data)
    
    def __getitem__(self, idx):
        '''
        Returns the training/validation/test example as index idx.
        '''

        curr_data = self.data[idx]
        # Keep inference prompt format consistent with training: "MR || REF".
        prompt_text = curr_data + " " + DELIMITER
        data_tokens = tokenize_text(prompt_text, tokenizer, padding=False)
        input_ids = data_tokens["input_ids"].squeeze()
        attention_mask = data_tokens["attention_mask"].squeeze()
        # prompt_len = len(tokenizer.encode(prompt_text, add_special_tokens=False, return_tensors="pt").squeeze())
        prompt_len = len(prompt_text)
        
        inputs = {
            'input_ids' : input_ids.to(dtype=torch.long),
            'attention_mask' : attention_mask.to(dtype=torch.long),
            'prompt_len' : prompt_len
        }
        return inputs
    




In [ ]:
unique_raw_mrs, multi_refs = data_retrieval(testwref_df, unique_mr=True)
unique_raw_mrs_dataset = E2EGenerationDataset(unique_raw_mrs)

In [ ]:
def generate(model, tokenizer, dataset):
    model.eval()

    preds = []

    # dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)
    dataloader = DataLoader(dataset, batch_size=1)


    for batch_idx, data in tqdm(enumerate(dataloader), total=len(dataloader)):
        mr_ids = data["input_ids"].to(device)
        mr_attention_mask = data["attention_mask"].to(device)
        
        with torch.no_grad():

            output = model.generate(
                input_ids=mr_ids,
                attention_mask=mr_attention_mask,
                max_new_tokens=CFG['max_new_tokens'],
                num_beams=CFG['beam_size'],
                length_penalty=CFG['length_penalty'],
                no_repeat_ngram_size=CFG['no_repeat_ngram_size'],
                repetition_penalty=1.0,
                early_stopping=True,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        

        # # DELIMITER BASED MR REMOVAL ====================
        decoded = tokenizer.decode(output[0], skip_special_tokens=False)
        text = decoded
        text_cleaned = decoded

        # identify where prediction begins
        if DELIMITER in text_cleaned:
            text_cleaned = text_cleaned.split(DELIMITER, 1)[1].strip()

        # Truncate any trailing content after EOS marker.
        if tokenizer.eos_token in text_cleaned:
            text_cleaned = text_cleaned.split(tokenizer.eos_token, 1)[0].strip()

        # identify where prediction begins
        if DELIMITER in text_cleaned:
            text_cleaned = text_cleaned.replace(DELIMITER, "").strip()

        if tokenizer.pad_token in text_cleaned:
            text_cleaned = text_cleaned.replace(tokenizer.pad_token, "").strip()
            
        preds.append(text_cleaned)



        # TO EXPLORE GENERATIONS =====================
        # print("BATCH: \n", batch_idx)
        # print("Raw Decoded: \n", decoded)
        # print("Trimmed: \n", text)
        # print("Cleaned: \n",text_cleaned)
        # print("Decoded MR1: \n",tokenizer.decode(mr_ids))
        # print("Decoded MR2: \n",tokenizer.decode(mr_ids,skip_special_tokens=True))

        # if batch_idx == 10:
        #     break

    model.train()
    return preds

In [ ]:
unique_preds = generate(model, tokenizer, unique_raw_mrs_dataset)
for i in range(10):
    print(f"{i}'th MR's refs: {multi_refs[i]}")

In [ ]:
# Unified E2E metric evaluation
# Expects:
# - unique_preds: List[str] generated from unique MRs
# - multi_refs: List[List[str]] references for each unique MR


def _safe_tokenize_text(s: str):
    return s.strip().split()


def compute_e2e_metrics(predictions, multi_references):
    """
    Compute BLEU, NIST, METEOR, ROUGE-L, and CIDEr for E2E NLG.

    Args:
        predictions: List[str], one prediction per unique MR.
        multi_references: List[List[str]], list of refs for each MR.

    Returns:
        Dict[str, float] with keys: BLEU, NIST, METEOR, ROUGE_L, CIDEr.
    """
    assert len(predictions) == len(multi_references), (
        f"Mismatched lengths: {len(predictions)} predictions vs "
        f"{len(multi_references)} reference groups"
    )

    # sacrebleu expects refs grouped by reference index: List[List[str]]
    max_refs = max(len(refs) for refs in multi_references)
    sacre_refs = []
    for ref_idx in range(max_refs):
        ref_stream = []
        for refs in multi_references:
            if ref_idx < len(refs):
                ref_stream.append(refs[ref_idx])
            else:
                ref_stream.append("")
        sacre_refs.append(ref_stream)

    bleu_score = sacrebleu.corpus_bleu(predictions, sacre_refs).score / 100.0

    nist_hyps = [_safe_tokenize_text(p) for p in predictions]
    nist_refs = [[_safe_tokenize_text(r) for r in refs] for refs in multi_references]
    nist_score = corpus_nist(nist_refs, nist_hyps)

    # Build COCO-style dicts once: id -> [captions].
    cider_hyps = {}
    for i, pred in enumerate(predictions):
        cider_hyps[i] = [pred]

    cider_refs = {}
    for i, refs in enumerate(multi_references):
        cider_refs[i] = refs

    # METEOR with multi-reference input (COCO-style scorer).
    meteor_scorer = Meteor()
    meteor_score, _ = meteor_scorer.compute_score(cider_refs, cider_hyps)

    # ROUGE-L with multi-reference input (COCO-style scorer).
    rouge_scorer = Rouge()
    rouge_l, _ = rouge_scorer.compute_score(cider_refs, cider_hyps)

    # CIDEr uses COCO-style dicts: id -> [captions].
    cider_scorer = Cider()
    cider_score, _ = cider_scorer.compute_score(cider_refs, cider_hyps)

    return {
        "BLEU": float(bleu_score) * 100,
        "NIST": float(nist_score),
        "METEOR": float(meteor_score) * 100,
        "ROUGE_L": float(rouge_l) * 100,
        "CIDEr": float(cider_score),
    }

In [ ]:
results = compute_e2e_metrics(unique_preds, multi_refs)
for metric_name, metric_value in results.items():
    print(f"{metric_name}: {metric_value:.6f}")